# Stock Trading Recommending System using enhanced ML with core in Peak Valley Agorithm

Built a smart trading system for AAPL stock using a machine learning model called LSTM to predict prices and make buy/sell decisions. The core Peak Valley Algorithm spots key price highs and lows to trigger trades. Pulled stock data from Yahoo Finance, cleaned it with Pandas, and trained the model. Created a user-friendly Streamlit dashboard with light/dark modes, showing profit trends, portfolio growth, stock holdings, and trade signals through charts. Added a slick Stocks Tracker with cards for trades, profits, and a win rate gauge


In [1]:
import requests
import pandas as pd
import time
from datetime import datetime

## Step 1 Data Fetching

In [2]:
def fetch_stock_data(ticker, api_key, start_date='2024-05-01', end_date='2025-05-01'):
    """
    Fetches hourly stock data (Date, Open, Close, Volume) from Polygon.io with pagination.
    Args:
        ticker (str): Stock symbol (e.g., 'AAPL').
        api_key (str): Polygon.io API key.
        start_date (str): Start date (YYYY-MM-DD).
        end_date (str): End date (YYYY-MM-DD).
    Returns:
        pd.DataFrame: Fetched data or None if failed.
    """
    # Construct base URL
    url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/hour/{start_date}/{end_date}"
    params = {
        'adjusted': 'true',
        'sort': 'asc',
        'limit': 50000,  # Max rows per request
        'apiKey': api_key
    }
    
    all_data = []
    max_retries = 3
    
    while url:
        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params)
                response.raise_for_status()
                data = response.json()
                
                if 'results' not in data or not data['results']:
                    print(f"No data returned for {ticker}. Check date range or API key.")
                    return None
                
                # Append results to all_data
                all_data.extend(data['results'])
                
                # Check for pagination (next_url if more data exists)
                url = data.get('next_url', None)
                if url:
                    params = {'apiKey': api_key}  # Reset params for next_url
                break
                
            except requests.RequestException as e:
                print(f"API error on attempt {attempt + 1}: {e}")
                if attempt < max_retries - 1:
                    time.sleep(2)
                else:
                    print("Max retries reached.")
                    return None
            except KeyError as e:
                print(f"Data parsing error: {e}.")
                return None
    
    if not all_data:
        print("No data collected.")
        return None
    
    # Convert to DataFrame
    df = pd.DataFrame(all_data)
    
    # Select and rename columns
    df['Date'] = pd.to_datetime(df['t'], unit='ms')
    df = df[['Date', 'o', 'c', 'v']].rename(columns={
        'o': 'Open',
        'c': 'Close',
        'v': 'Volume'
    })
    
    # Validate recent price (AAPL ~$196.25 on 2025-05-07)
    latest_price = df['Close'].iloc[-1]
    expected_price = 196.25
    if abs(latest_price - expected_price) > 5:
        print(f"Warning: Latest price {latest_price} differs from expected {expected_price}")
    
    # Save to CSV
    csv_file = f'raw_{ticker.lower()}.csv'
    df.to_csv(csv_file, index=False)
    print(f"Saved {len(df)} rows to {csv_file}")
    return df

# Usage
if __name__ == "__main__":
    api_key = 'dgzMvXlGyL5viM07VZvFLTFIxeBPYY9n'
    df = fetch_stock_data('AAPL', api_key)
    if df is not None:
        print(df.tail())
    else:
        print("Failed to fetch data.")

Saved 4001 rows to raw_aapl.csv
                    Date      Open     Close     Volume
3996 2025-05-01 19:00:00  212.5500  212.8300  9534210.0
3997 2025-05-01 20:00:00  213.3200  207.7001  4431344.0
3998 2025-05-01 21:00:00  207.7992  205.2100  2686654.0
3999 2025-05-01 22:00:00  205.2500  204.5500   628233.0
4000 2025-05-01 23:00:00  204.5000  205.2500   467554.0


In [3]:
df

,Date,Open,Close,Volume
0,2024-05-01 08:00:00,170.5000,169.7900,23770.0
1,2024-05-01 09:00:00,169.7300,169.5900,13685.0
2,2024-05-01 10:00:00,169.5000,170.1000,24119.0
3,2024-05-01 11:00:00,170.2000,169.9900,36022.0
4,2024-05-01 12:00:00,170.1500,169.9702,259260.0
...,...,...,...,...
3996,2025-05-01 19:00:00,212.5500,212.8300,9534210.0
3997,2025-05-01 20:00:00,213.3200,207.7001,4431344.0
3998,2025-05-01 21:00:00,207.7992,205.2100,2686654.0
3999,2025-05-01 22:00:00,205.2500,204.5500,628233.0


### Cleaning and Processing

In [4]:


def clean_stock_data(input_file='raw_aapl.csv', output_file='cleaned_aapl.csv'):
    """
    Cleans raw stock data by handling missing values, ensuring datetime, removing duplicates,
    and saving to a lightweight CSV.
    Args:
        input_file (str): Path to raw CSV (e.g., 'raw_aapl.csv').
        output_file (str): Path to save cleaned CSV (e.g., 'cleaned_aapl.csv').
    Returns:
        pd.DataFrame: Cleaned data or None if failed.
    """
    try:
        # Load raw CSV
        df = pd.read_csv(input_file)
        print(f"Loaded {len(df)} rows from {input_file}")
        
        # Ensure expected columns
        expected_columns = ['Date', 'Open', 'Close', 'Volume']
        if not all(col in df.columns for col in expected_columns):
            print(f"Error: Missing columns. Found: {df.columns}")
            return None
        
        # Ensure Date is datetime
        df['Date'] = pd.to_datetime(df['Date'])
        
        # Forward fill missing values
        df[['Open', 'Close', 'Volume']] = df[['Open', 'Close', 'Volume']].fillna(method='ffill')
        
        # Check for remaining missing values
        if df[['Open', 'Close', 'Volume']].isna().any().any():
            print("Warning: Some missing values remain. Dropping affected rows.")
            df = df.dropna(subset=['Open', 'Close', 'Volume'])
        
        # Remove duplicates
        initial_rows = len(df)
        df = df.drop_duplicates(subset=['Date'], keep='first')
        if len(df) < initial_rows:
            print(f"Removed {initial_rows - len(df)} duplicates")
        
        # Select columns
        df = df[['Date', 'Open', 'Close', 'Volume']]
        
        # Validate data
        if len(df) < 1000:
            print(f"Warning: Only {len(df)} rows. Expected ~1,638 rows.")
        latest_price = df['Close'].iloc[-1]
        expected_price = 196.25
        if abs(latest_price - expected_price) > 5:
            print(f"Warning: Latest price {latest_price} differs from expected {expected_price}")
        
        # Save to CSV
        df.to_csv(output_file, index=False)
        print(f"Saved {len(df)} rows to {output_file}")
        return df
        
    except FileNotFoundError:
        print(f"Error: {input_file} not found.")
        return None
    except pd.errors.EmptyDataError:
        print(f"Error: {input_file} is empty.")
        return None
    except Exception as e:
        print(f"Cleaning error: {e}")
        return None

# Usage
if __name__ == "__main__":
    df_clean = clean_stock_data()
    if df_clean is not None:
        print(df_clean.tail())
    else:
        print("Failed to clean data.")

Loaded 4001 rows from raw_aapl.csv
Saved 4001 rows to cleaned_aapl.csv
                    Date      Open     Close     Volume
3996 2025-05-01 19:00:00  212.5500  212.8300  9534210.0
3997 2025-05-01 20:00:00  213.3200  207.7001  4431344.0
3998 2025-05-01 21:00:00  207.7992  205.2100  2686654.0
3999 2025-05-01 22:00:00  205.2500  204.5500   628233.0
4000 2025-05-01 23:00:00  204.5000  205.2500   467554.0


In [5]:
df1 = pd.read_csv(r"C:\Users\abhay\Stock Trading Recommendation System\cleaned_aapl.csv")

In [6]:
df1

,Date,Open,Close,Volume
0,2024-05-01 08:00:00,170.5000,169.7900,23770.0
1,2024-05-01 09:00:00,169.7300,169.5900,13685.0
2,2024-05-01 10:00:00,169.5000,170.1000,24119.0
3,2024-05-01 11:00:00,170.2000,169.9900,36022.0
4,2024-05-01 12:00:00,170.1500,169.9702,259260.0
...,...,...,...,...
3996,2025-05-01 19:00:00,212.5500,212.8300,9534210.0
3997,2025-05-01 20:00:00,213.3200,207.7001,4431344.0
3998,2025-05-01 21:00:00,207.7992,205.2100,2686654.0
3999,2025-05-01 22:00:00,205.2500,204.5500,628233.0


## Feature Engineering 

Feature engineering is the process of creating and selecting meaningful input variables (features) from raw data to improve machine learning model performance, crucial for your Stock Trading Recommendation System.
Transform raw AAPL stock data into predictive features that enhance your LSTM and Random Forest models.


In [7]:
# create a output file containing moving averages and predicted values (hourly) using random forest.

def create_features(input_file='cleaned_aapl.csv', output_file='features_aapl.csv'):
    """
    Creates features (5-hour MA, 10-hour MA, hourly returns) from cleaned stock data.
    Args:
        input_file (str): Path to cleaned CSV (e.g., 'cleaned_aapl.csv').
        output_file (str): Path to save features CSV (e.g., 'features_aapl.csv').
    Returns:
        pd.DataFrame: Data with features or None if failed.
    """
    try:
        # Load cleaned CSV
        df = pd.read_csv(input_file)
        print(f"Loaded {len(df)} rows from {input_file}")
        
        # Ensure expected columns
        expected_columns = ['Date', 'Open', 'Close', 'Volume']
        if not all(col in df.columns for col in expected_columns):
            print(f"Error: Missing columns. Found: {df.columns}")
            return None
        
        # Ensure Date is datetime
        df['Date'] = pd.to_datetime(df['Date'])
        
        # Compute 5-hour and 10-hour moving averages for Close
        df['MA5'] = df['Close'].rolling(window=5).mean()
        df['MA10'] = df['Close'].rolling(window=10).mean()
        
        # Compute hourly returns: (Close - Previous Close) / Previous Close
        df['Return'] = (df['Close'] - df['Close'].shift(1)) / df['Close'].shift(1)
        
        # Handle missing values (MA5: first 4 rows NaN, MA10: first 9, Return: first 1)
        df[['MA5', 'MA10', 'Return']] = df[['MA5', 'MA10', 'Return']].fillna(method='ffill')
        
        # Drop any remaining NaN rows
        if df[['MA5', 'MA10', 'Return']].isna().any().any():
            print("Warning: Dropping rows with remaining missing feature values.")
            df = df.dropna(subset=['MA5', 'MA10', 'Return'])
        
        # Select output columns
        df = df[['Date', 'Open', 'Close', 'Volume', 'MA5', 'MA10', 'Return']]
        
        # Validate features
        if len(df) < 1000:
            print(f"Warning: Only {len(df)} rows. Expected ~1,638.")
        if df['Return'].abs().max() > 0.1:
            print(f"Warning: Large returns (max: {df['Return'].abs().max():.4f}).")
        
        # Save to CSV
        df.to_csv(output_file, index=False)
        print(f"Saved {len(df)} rows to {output_file}")
        return df
        
    except FileNotFoundError:
        print(f"Error: {input_file} not found.")
        return None
    except pd.errors.EmptyDataError:
        print(f"Error: {input_file} is empty.")
        return None
    except Exception as e:
        print(f"Feature engineering error: {e}")
        return None

# Usage
if __name__ == "__main__":
    df_features = create_features()
    if df_features is not None:
        print(df_features.tail())
    else:
        print("Failed to create features.")

Loaded 4001 rows from cleaned_aapl.csv
Saved 3992 rows to features_aapl.csv
                    Date      Open     Close     Volume        MA5       MA10  \
3996 2025-05-01 19:00:00  212.5500  212.8300  9534210.0  212.33788  211.36894   
3997 2025-05-01 20:00:00  213.3200  207.7001  4431344.0  211.56102  211.13995   
3998 2025-05-01 21:00:00  207.7992  205.2100  2686654.0  210.17602  210.72995   
3999 2025-05-01 22:00:00  205.2500  204.5500   628233.0  208.56702  210.27195   
4000 2025-05-01 23:00:00  204.5000  205.2500   467554.0  207.10802  209.72895   

        Return  
3996  0.001341  
3997 -0.024103  
3998 -0.011989  
3999 -0.003216  
4000  0.003422  


## Random Forest For Prective Analysis

Random Forest is an ensemble machine learning algorithm used for predictive analysis, ideal for the Stock Trading Recommendation System due to its robustness and accuracy. It combines multiple decision trees, each trained on random subsets of data and features, to make predictions by averaging.


In [8]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def train_predict_model(input_file='features_aapl.csv', output_file='predictions_aapl.csv'):
    """
    Trains Random Forest to predict next hour's Close price and saves predictions.
    Args:
        input_file (str): Path to features CSV (e.g., 'features_aapl.csv').
        output_file (str): Path to save predictions CSV (e.g., 'predictions_aapl.csv').
    Returns:
        pd.DataFrame: Predictions or None if failed.
    """
    try:
        # Load features CSV
        df = pd.read_csv(input_file)
        print(f"Loaded {len(df)} rows from {input_file}")
        
        # Ensure expected columns
        expected_columns = ['Date', 'Close', 'MA5', 'MA10', 'Return']
        if not all(col in df.columns for col in expected_columns):
            print(f"Error: Missing columns. Found: {df.columns}")
            return None
        
        # Prepare features and target
        features = ['Close', 'MA5', 'MA10', 'Return']
        df['Target'] = df['Close'].shift(-1)  # Next hour's Close
        df = df.dropna()  # Drop rows with NaN Target (last row)
        
        # Split data: 80% train, 20% test (no shuffle for time-series)
        train_size = int(0.8 * len(df))
        train_df = df.iloc[:train_size]
        test_df = df.iloc[train_size:]
        
        X_train = train_df[features]
        y_train = train_df['Target']
        X_test = test_df[features]
        y_test = test_df['Target']
        
        # Train Random Forest
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)
        
        # Predict on test set
        predictions = model.predict(X_test)
        
        # Calculate Mean Absolute Error
        mae = mean_absolute_error(y_test, predictions)
        print(f"Mean Absolute Error: ${mae:.2f}")
        
        # Create predictions DataFrame
        results = pd.DataFrame({
            'Date': test_df['Date'],
            'Actual_Close': y_test,
            'Predicted_Close': predictions
        })
        
        # Validate predictions
        if mae > 5:  # Arbitrary threshold for hourly price
            print(f"Warning: High MAE (${mae:.2f}). Model may need tuning.")
        if len(results) < 100:
            print(f"Warning: Only {len(results)} predictions. Expected ~328.")
        
        # Save to CSV
        results.to_csv(output_file, index=False)
        print(f"Saved {len(results)} rows to {output_file}")
        return results
        
    except FileNotFoundError:
        print(f"Error: {input_file} not found.")
        return None
    except pd.errors.EmptyDataError:
        print(f"Error: {input_file} is empty.")
#         except Exception as e:
#         print(f"ML prediction error: {e}")
        return None

# Usage
if __name__ == "__main__":
    df_predictions = train_predict_model()
    if df_predictions is not None:
        print(df_predictions.tail())
    else:
        print("Failed to generate predictions.")

Loaded 3992 rows from features_aapl.csv
Mean Absolute Error: $1.39
Saved 799 rows to predictions_aapl.csv
                     Date  Actual_Close  Predicted_Close
3986  2025-05-01 18:00:00      212.8300       212.985684
3987  2025-05-01 19:00:00      207.7001       213.221856
3988  2025-05-01 20:00:00      205.2100       207.056349
3989  2025-05-01 21:00:00      204.5500       205.808605
3990  2025-05-01 22:00:00      205.2500       205.271582


#  Peak Valley Core 
The Peak Valley Algorithm, central to your Stock Trading Recommendation System, is a heuristic approach designed to identify critical turning points in AAPL stock price movements by detecting local maxima (peaks) and minima (valleys) within time-series data. In the context of your project, it analyzes historical price data (e.g., Close prices from trades_aapl.csv) to pinpoint high and low points based on predefined thresholds, such as price changes exceeding a set percentage or sustained trends over a window of time


In [9]:
import pandas as pd
from datetime import datetime

class PeakValleyTrader:
    def __init__(self, input_file='predictions_aapl.csv', initial_cash=10000):
        self.df = pd.read_csv(input_file)
        self.df['Date'] = pd.to_datetime(self.df['Date'])
        self.initial_cash = initial_cash
        self.cash = initial_cash
        self.shares = 0
        self.trades = []
        self.portfolio = []
        self.transaction_fee = 0.001  # 0.1% per trade
        self.min_profit_hourly = 0.10  # Min profit per trade (hourly)
        self.min_profit_daily = 1.00   # Min profit per trade (daily)
        
    def aggregate_daily(self):
        """Aggregate hourly data to daily (last Close of day)."""
        return self.df.resample('D', on='Date').agg({
            'Predicted_Close': 'last',
            'Actual_Close': 'last'
        }).dropna().reset_index()
    
    def peak_valley(self, prices, dates, time_frame='hourly'):
        """Run peak-valley algorithm for given prices and time frame."""
        max_wait = 24 if time_frame == 'hourly' else 5  # Hours or days
        min_profit = self.min_profit_hourly if time_frame == 'hourly' else self.min_profit_daily
        i = 0
        while i < len(prices) - 1:
            # Find valley
            valley_start = i
            while i < len(prices) - 1 and prices[i] >= prices[i + 1]:
                i += 1
            if i >= len(prices) - 1 or i - valley_start > max_wait:
                break
            buy_index = i
            buy_price = prices[i]
            buy_date = dates[i]
            
            # Find peak
            peak_start = i
            while i < len(prices) - 1 and prices[i] <= prices[i + 1]:
                i += 1
            if i - peak_start > max_wait:
                continue
            sell_index = i
            sell_price = prices[i]
            sell_date = dates[i]
            
            # Calculate profit
            profit = sell_price - buy_price
            if profit > min_profit:
                fee = (buy_price + sell_price) * self.transaction_fee
                net_profit = profit - fee
                if net_profit > 0:
                    self.trades.append({
                        'Buy_Date': buy_date,
                        'Buy_Price': buy_price,
                        'Sell_Date': sell_date,
                        'Sell_Price': sell_price,
                        'Profit': net_profit,
                        'Time_Frame': time_frame
                    })
                    # Update portfolio
                    shares_bought = (self.cash * 0.5) // buy_price  # Use half cash
                    if shares_bought > 0:
                        self.cash -= shares_bought * buy_price * (1 + self.transaction_fee)
                        self.shares += shares_bought
                        self.cash += shares_bought * sell_price * (1 - self.transaction_fee)
                        self.shares -= shares_bought
                        self.portfolio.append({
                            'Date': sell_date,
                            'Cash': self.cash,
                            'Shares': self.shares,
                            'Portfolio_Value': self.cash + self.shares * sell_price
                        })
    
    def query_trade(self, current_date, time_frame='hourly'):
        """Provide buy/sell recommendation for a given date."""
        df = self.df if time_frame == 'hourly' else self.aggregate_daily()
        current_date = pd.to_datetime(current_date)
        idx = df[df['Date'] <= current_date].index
        if not idx.empty:
            i = idx[-1]
            prices = df['Predicted_Close'].values
            if i > 0 and prices[i] < prices[i-1] and (i == len(prices)-1 or prices[i] < prices[i+1]):
                return f"Buy at {df['Date'].iloc[i]} (price: ${prices[i]:.2f}) - potential valley."
            elif i > 0 and prices[i] > prices[i-1] and (i == len(prices)-1 or prices[i] > prices[i+1]):
                return f"Sell at {df['Date'].iloc[i]} (price: ${prices[i]:.2f}) - potential peak."
        return "Hold - no clear valley or peak."
    
    def portfolio_verdict(self, trades_df, portfolio_df):
        """Generate portfolio report verdict."""
        if trades_df.empty or portfolio_df.empty:
            return "Portfolio Verdict: No trades executed, no performance to evaluate."
        
        total_profit = trades_df['Profit'].sum()
        win_rate = (trades_df['Profit'] > 0).mean() * 100
        final_value = portfolio_df['Portfolio_Value'].iloc[-1]
        growth = ((final_value - self.initial_cash) / self.initial_cash) * 100
        
        if growth > 2:
            performance = "Strong"
        elif growth > 0:
            performance = "Moderate"
        else:
            performance = "Loss"
        
        return (f"Portfolio Verdict: Total Profit: ${total_profit:.2f}, "
                f"Win Rate: {win_rate:.1f}%, Growth: {growth:.2f}%, "
                f"Performance: {performance}")
    
    def run(self, output_trades='trades_aapl.csv', output_portfolio='portfolio_aapl.csv'):
        """Run peak-valley, save trades/portfolio, and print verdict."""
        try:
            # Run for hourly and daily
            self.peak_valley(self.df['Predicted_Close'].values, self.df['Date'].values, 'hourly')
            daily_df = self.aggregate_daily()
            self.peak_valley(daily_df['Predicted_Close'].values, daily_df['Date'].values, 'daily')
            
            # Save trades
            trades_df = pd.DataFrame(self.trades)
            if trades_df.empty:
                print("No profitable trades found.")
                return None
            
            # Save portfolio
            portfolio_df = pd.DataFrame(self.portfolio)
            if portfolio_df.empty:
                print("No portfolio updates recorded.")
                return None
            
            # Save CSVs
            trades_df.to_csv(output_trades, index=False)
            portfolio_df.to_csv(output_portfolio, index=False)
            
            # Print results
            print(f"Saved {len(trades_df)} trades to {output_trades}")
            print(f"Saved {len(portfolio_df)} portfolio entries to {output_portfolio}")
            print(f"Final Portfolio Value: ${portfolio_df['Portfolio_Value'].iloc[-1]:.2f}")
            print(self.portfolio_verdict(trades_df, portfolio_df))
            return trades_df, portfolio_df
            
        except Exception as e:
            print(f"Error: {e}")
            return None

# Usage
if __name__ == "__main__":
    trader = PeakValleyTrader()
    trades_df, portfolio_df = trader.run()
    if trades_df is not None:
        print(trades_df)
    # Example interactive query
    print(trader.query_trade('2025-05-07 15:00:00', 'hourly'))

Saved 176 trades to trades_aapl.csv
Saved 176 portfolio entries to portfolio_aapl.csv
Final Portfolio Value: $25783.08
Portfolio Verdict: Total Profit: $395.30, Win Rate: 100.0%, Growth: 157.83%, Performance: Strong
               Buy_Date   Buy_Price           Sell_Date  Sell_Price  \
0   2025-02-20 13:00:00  243.979156 2025-02-20 14:00:00  244.486748   
1   2025-02-20 15:00:00  244.410025 2025-02-20 20:00:00  246.176278   
2   2025-02-21 11:00:00  244.784372 2025-02-21 13:00:00  246.155830   
3   2025-02-21 14:00:00  245.569308 2025-02-21 15:00:00  246.199596   
4   2025-02-21 16:00:00  246.134134 2025-02-21 17:00:00  248.950362   
..                  ...         ...                 ...         ...   
171 2025-03-26 00:00:00  221.467884 2025-03-27 00:00:00  223.548125   
172 2025-03-28 00:00:00  217.393873 2025-04-01 00:00:00  223.440121   
173 2025-04-08 00:00:00  176.160575 2025-04-09 00:00:00  202.727170   
174 2025-04-10 00:00:00  189.315484 2025-04-14 00:00:00  205.987477   
175

### Step 6: ML Enhancement with Email Alerts
Enhance Random Forest with RSI, Volume_MA5, Volatility, tune hyperparameters, and send email alerts for buy/sell signals.

## Enhanced ML

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import smtplib
from email.mime.text import MIMEText
from datetime import datetime

# Load data
df = pd.read_csv('features_aapl.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Add features
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(window=14).mean()
loss = -delta.where(delta < 0, 0).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))
df['Volume_MA5'] = df['Volume'].rolling(window=5).mean()
df['Volatility'] = df['Return'].rolling(window=5).std()
df = df.dropna()
df.to_csv('features_enhanced_aapl.csv', index=False)
print(f"Saved {len(df)} rows to features_enhanced_aapl.csv")

# Train model
features = ['Close', 'MA5', 'MA10', 'Return', 'RSI', 'Volume_MA5', 'Volatility']
df['Target'] = df['Close'].shift(-1)
df = df.dropna()
train_size = int(0.8 * len(df))
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]
X_train = train_df[features]
y_train = train_df['Target']
X_test = test_df[features]
y_test = test_df['Target']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Scaled feature means:", X_train_scaled.mean(axis=0))  # Should be ~0

param_grid = {'n_estimators': [100, 200, 300], 'max_depth': [5, 10, 15]}
model = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(model, param_grid, cv=3, scoring='neg_mean_absolute_error')
grid_search.fit(X_train_scaled, y_train)

model = grid_search.best_estimator_
predictions = model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, predictions)
feature_importance = pd.Series(model.feature_importances_, index=features)

results = pd.DataFrame({
    'Date': test_df['Date'].reset_index(drop=True),
    'Actual_Close': y_test.reset_index(drop=True),
    'Predicted_Close': predictions
})
results['Date'] = pd.to_datetime(results['Date'])
results.to_csv('predictions_enhanced_aapl.csv', index=False)
print(f"Saved {len(results)} predictions to predictions_enhanced_aapl.csv")
print(f"Enhanced MAE: ${mae:.2f}")
print("Feature Importance:\n", feature_importance)

# Email alert function
def send_email_alert(trade, email_config):
    try:
        subject = f"Trade Alert: {'Buy' if trade['Profit'] >= 0 else 'Sell'} Signal"
        body = (f"Trade Recommendation:\n"
                f"Action: {'Buy' if trade['Profit'] >= 0 else 'Sell'}\n"
                f"Date: {trade['Buy_Date'] if trade['Profit'] >= 0 else trade['Sell_Date']}\n"
                f"Price: ${trade['Buy_Price'] if trade['Profit'] >= 0 else trade['Sell_Price']:.2f}\n"
                f"Confidence: {trade['Confidence']:.2f}\n"
                f"Portfolio Value: ${trade['Portfolio_Value']:.2f}")
        msg = MIMEText(body)
        msg['Subject'] = subject
        msg['From'] = email_config['sender_email']
        msg['To'] = email_config['receiver_email']
        
        with smtplib.SMTP(email_config['smtp_server'], email_config['smtp_port']) as server:
            server.starttls()
            server.login(email_config['sender_email'], email_config['sender_password'])
            server.send_message(msg)
        print(f"Email sent for trade at {trade['Buy_Date']}")
    except Exception as e:
        print(f"Email error: {e}")

# Peak-valley
prices = results['Predicted_Close'].values
dates = results['Date']  # Use pd.Series of pd.Timestamp
cash = 10000
shares = 0
trades = []
portfolio = []
daily_trades = {}
print("Date type check:", type(dates.iloc[0]))  # Should be pd.Timestamp
i = 0
while i < len(prices) - 1:
    valley_start = i
    while i < len(prices) - 1 and prices[i] >= prices[i + 1] * 1.002:
        i += 1
    if i >= len(prices) - 1 or i - valley_start > 96:
        break
    buy_index = i
    buy_price = prices[i]
    buy_date = dates.iloc[i]
    
    buy_day = buy_date.strftime('%Y-%m-%d')
    daily_trades[buy_day] = daily_trades.get(buy_day, 0) + 1
    if daily_trades[buy_day] > 2:
        i += 1
        continue
    
    peak_start = i
    for j in range(i, min(i + 97, len(prices))):
        i = j
        current_price = prices[i]
        if current_price < buy_price * 0.98:
            sell_price = current_price
            sell_date = dates.iloc[j]
            break
        if j < len(prices) - 1 and prices[j] > prices[j + 1] * 1.002:
            sell_price = prices[j]
            sell_date = dates.iloc[j]
            break
    else:
        continue
    
    profit = sell_price - buy_price
    if profit > 0.10 or profit < -buy_price * 0.02:
        fee = (buy_price + sell_price) * 0.001
        net_profit = profit - fee
        confidence = min(1.0, (abs(profit) / buy_price) / (mae / buy_price))
        shares_bought = (cash * 0.5) // buy_price
        if shares_bought > 0:
            cash -= shares_bought * buy_price * 1.001
            shares += shares_bought
            cash += shares_bought * sell_price * 0.999
            shares -= shares_bought
            portfolio_value = cash + shares * sell_price
            trade = {
                'Buy_Date': buy_date,
                'Buy_Price': buy_price,
                'Sell_Date': sell_date,
                'Sell_Price': sell_price,
                'Profit': net_profit,
                'Time_Frame': 'hourly',
                'Confidence': confidence,
                'Portfolio_Value': portfolio_value
            }
            trades.append(trade)
            portfolio.append({
                'Date': sell_date,
                'Cash': cash,
                'Shares': shares,
                'Portfolio_Value': portfolio_value
            })
            email_config = {
                'smtp_server': 'smtp.gmail.com',
                'smtp_port': 587,
                'sender_email': 'abhaykush050804@gmail.com',
                'sender_password': 'kckjuyxwyjycalye',
                'receiver_email': 'akks1925@gmail.com'
            }
            send_email_alert(trade, email_config)

trades_df = pd.DataFrame(trades)
portfolio_df = pd.DataFrame(portfolio)
if not trades_df.empty:
    trades_df.to_csv('trades_aapl.csv', index=False)
    portfolio_df.to_csv('portfolio_aapl.csv', index=False)
    print(f"Saved {len(trades_df)} trades to trades_aapl.csv")
    print(f"Saved {len(portfolio_df)} portfolio entries to portfolio_aapl.csv")

# Portfolio verdict
if not trades_df.empty:
    total_profit = trades_df['Profit'].sum()
    win_rate = (trades_df['Profit'] > 0).mean() * 100
    final_value = portfolio_df['Portfolio_Value'].iloc[-1]
    growth = ((final_value - 10000) / 10000) * 100
    cash_ratio = portfolio_df['Cash'].iloc[-1] / final_value
    diversification = "Balanced" if 0.4 <= cash_ratio <= 0.6 else "Unbalanced"
    returns = trades_df['Profit'] / trades_df['Buy_Price']
    sharpe_ratio = np.mean(returns) / np.std(returns) * np.sqrt(252 * 6)
    performance = "Strong" if growth > 2 else "Moderate" if growth > 0 else "Loss"
    print(f"Portfolio Verdict: Total Profit: ${total_profit:.2f}, "
          f"Win Rate: {win_rate:.1f}%, Growth: {growth:.2f}%, "
          f"Diversification: {diversification}, Sharpe Ratio: {sharpe_ratio:.2f}, "
          f"Performance: {performance}")
else:
    print("Portfolio Verdict: No trades executed.")

## Step 7: Streamlit Dashboard

Visualize ML-predicted prices, trading signals, and portfolio performance with an interactive Streamlit dashboard.
Developed a robust trading platform for AAPL, leveraging an LSTM-based ML model for precise stock price forecasting and automated buy/sell signal generation. Utilized Yahoo Finance data, cleaned and processed with Pandas, to train the model. Designed a dynamic Streamlit dashboard with light/dark mode, featuring impactful visualizations: profit margin trends, portfolio growth, stock holdings bar charts, and profit share area charts. Enhanced UX with a sidebar navbar, date filters, and collapsible graph sections. Implemented a sleek Stocks Tracker with cards for trades, profits, signals, and a win rate gauge

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import numpy as np

# Page config (must be first Streamlit command)
st.set_page_config(page_title="AAPL Trading Dashboard", layout="wide")

# Initialize session state for theme
if 'theme' not in st.session_state:
    st.session_state.theme = 'light'

# Custom CSS for light and dark themes, including cards
light_css = """
<style>
body, .stApp {
    background-color: #ffffff;
    color: #000000;
}
.stSidebar {
    background-color: #f0f2f6;
}
.stPlotlyChart, .stDataFrame {
    background-color: #ffffff;
    border: 1px solid #e6e6e6;
}
.theme-toggle {
    position: fixed;
    top: 10px;
    right: 10px;
    z-index: 1000;
}
h1, h2, h3, h4, h5, h6 {
    color: #000000 !important;
}
.card {
    background-color: #f0f2f6;
    border: 1px solid #e6e6e6;
    border-radius: 8px;
    padding: 10px;
    margin: 5px 0;
    text-align: center;
    font-size: 16px;
    font-weight: bold;
    color: #000000;
}
</style>
"""

dark_css = """
<style>
body, .stApp {
    background-color: #1e1e1e;
    color: #ffffff;
}
.stSidebar {
    background-color: #2c2c2c;
}
.stPlotlyChart, .stDataFrame {
    background-color: #2c2c2c;
    border: 1px solid #444444;
}
.theme-toggle {
    position: fixed;
    top: 10px;
    right: 10px;
    z-index: 1000;
}
h1, h2, h3, h4, h5, h6 {
    color: #ffffff !important;
}
.card {
    background-color: #2c2c2c;
    border: 1px solid #444444;
    border-radius: 8px;
    padding: 10px;
    margin: 5px 0;
    text-align: center;
    font-size: 16px;
    font-weight: bold;
    color: #ffffff;
}
</style>
"""

# Apply theme based on session state
if st.session_state.theme == 'dark':
    st.markdown(dark_css, unsafe_allow_html=True)
else:
    st.markdown(light_css, unsafe_allow_html=True)

# Theme toggle button in top-right corner
st.markdown(
    '<div class="theme-toggle">',
    unsafe_allow_html=True
)
if st.button(f"Switch to {'Light' if st.session_state.theme == 'dark' else 'Dark'} Mode"):
    st.session_state.theme = 'light' if st.session_state.theme == 'dark' else 'dark'
    st.rerun()
st.markdown('</div>', unsafe_allow_html=True)

# Title and header
st.title("Stock Trading Recommendation System: AAPL Dashboard")
st.markdown("Visualizing ML-Predicted Prices and Trading Performance")

# Load data
@st.cache_data
def load_data():
    try:
        predictions_df = pd.read_csv('predictions_enhanced_aapl.csv')
        trades_df = pd.read_csv('trades_aapl.csv')
        portfolio_df = pd.read_csv('portfolio_aapl.csv')
        predictions_df['Date'] = pd.to_datetime(predictions_df['Date'])
        trades_df['Buy_Date'] = pd.to_datetime(trades_df['Buy_Date'])
        trades_df['Sell_Date'] = pd.to_datetime(trades_df['Sell_Date'])
        portfolio_df['Date'] = pd.to_datetime(portfolio_df['Date'])
        return predictions_df, trades_df, portfolio_df
    except FileNotFoundError as e:
        st.error(f"Error: {e}. Ensure CSVs are in the project folder.")
        return None, None, None
    except Exception as e:
        st.error(f"Error loading data: {e}")
        return None, None, None

predictions_df, trades_df, portfolio_df = load_data()
if predictions_df is None:
    st.stop()

# Check trades_df columns
required_trade_cols = ['Buy_Date', 'Buy_Price', 'Sell_Date', 'Sell_Price', 'Profit', 'Time_Frame']
if not all(col in trades_df.columns for col in required_trade_cols):
    st.error("Error: trades_aapl.csv missing required columns: " + ", ".join(required_trade_cols))
    st.stop()

# Sidebar for interactivity
st.sidebar.header("Navigation")
nav_option = st.sidebar.radio(
    "Best Fit to See Graphs",
    ["Full Dashboard", "Graphs Only"]
)

st.sidebar.header("Filter Options")
view = st.sidebar.selectbox("Select View", ["Hourly", "Daily"])
date_range = st.sidebar.slider(
    "Select Date Range",
    min_value=predictions_df['Date'].min().to_pydatetime(),
    max_value=predictions_df['Date'].max().to_pydatetime(),
    value=(predictions_df['Date'].min().to_pydatetime(), predictions_df['Date'].max().to_pydatetime()),
    format="YYYY-MM-DD"
)

# Stocks Tracker in sidebar (cards and gauge)
st.sidebar.header("Stocks Tracker")
if not trades_df.empty:
    total_trades = len(trades_df)
    avg_profit = trades_df['Profit'].mean()
    total_profit = trades_df['Profit'].sum()
    win_rate = (trades_df['Profit'] > 0).mean() * 100
    filtered_buy_signals = len(trades_df[
        (trades_df['Buy_Date'] >= pd.to_datetime(date_range[0])) &
        (trades_df['Buy_Date'] <= pd.to_datetime(date_range[1]))
    ])
    filtered_sell_signals = len(trades_df[
        (trades_df['Sell_Date'] >= pd.to_datetime(date_range[0])) &
        (trades_df['Sell_Date'] <= pd.to_datetime(date_range[1]))
    ])
    
    # Cards
    st.sidebar.markdown(f'<div class="card">Total Trades: {total_trades}</div>', unsafe_allow_html=True)
    st.sidebar.markdown(f'<div class="card">Average Profit: ${avg_profit:.2f}</div>', unsafe_allow_html=True)
    st.sidebar.markdown(f'<div class="card">Total Profit: ${total_profit:.2f}</div>', unsafe_allow_html=True)
    st.sidebar.markdown(f'<div class="card">Buy Signals: {filtered_buy_signals}</div>', unsafe_allow_html=True)
    st.sidebar.markdown(f'<div class="card">Sell Signals: {filtered_sell_signals}</div>', unsafe_allow_html=True)
    
    # Gauge for Win Rate
    fig_gauge = go.Figure(go.Indicator(
        mode="gauge+number",
        value=win_rate,
        title={'text': "Win Rate (%)"},
        gauge={
            'axis': {'range': [0, 100], 'tickwidth': 1, 'tickcolor': "#000000" if st.session_state.theme == 'light' else "#ffffff"},
            'bar': {'color': "#1f77b4"},
            'bgcolor': "#ffffff" if st.session_state.theme == 'light' else "#2c2c2c",
            'bordercolor': "#e6e6e6" if st.session_state.theme == 'light' else "#444444"
        }
    ))
    fig_gauge.update_layout(
        font=dict(size=12, color='#000000' if st.session_state.theme == 'light' else '#ffffff'),
        margin=dict(l=10, r=10, t=50, b=10),
        paper_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c'
    )
    st.sidebar.plotly_chart(fig_gauge, use_container_width=True)
else:
    st.sidebar.warning("No trade data available.")

# Filter data by date range
filtered_predictions = predictions_df[
    (predictions_df['Date'] >= pd.to_datetime(date_range[0])) &
    (predictions_df['Date'] <= pd.to_datetime(date_range[1]))
]
filtered_trades = trades_df[
    (trades_df['Buy_Date'] >= pd.to_datetime(date_range[0])) &
    (trades_df['Sell_Date'] <= pd.to_datetime(date_range[1]))
]
filtered_portfolio = portfolio_df[
    (portfolio_df['Date'] >= pd.to_datetime(date_range[0])) &
    (portfolio_df['Date'] <= pd.to_datetime(date_range[1]))
]

# Aggregate for daily view
if view == "Daily" and not filtered_trades.empty:
    daily_trades = []
    grouped = filtered_trades.groupby(filtered_trades['Buy_Date'].dt.date)
    for date, group in grouped:
        sell_date = group['Sell_Date'].min() if not group['Sell_Date'].isna().all() else pd.to_datetime(date)
        daily_trade = {
            'Buy_Date': pd.to_datetime(date),
            'Buy_Price': group['Buy_Price'].mean(),
            'Sell_Price': group['Sell_Price'].mean(),
            'Profit': group['Profit'].sum(),
            'Time_Frame': group['Time_Frame'].iloc[0],
            'Sell_Date': sell_date
        }
        daily_trades.append(daily_trade)
    filtered_trades = pd.DataFrame(daily_trades)
    if filtered_trades['Sell_Date'].isna().any():
        st.warning("Some Sell_Date values are missing in daily view. Using Buy_Date as fallback.")
        filtered_trades['Sell_Date'] = filtered_trades['Sell_Date'].fillna(filtered_trades['Buy_Date'])
    
    filtered_predictions = filtered_predictions.resample('D', on='Date').mean().reset_index()
    filtered_predictions['Date'] = pd.to_datetime(filtered_predictions['Date'])
    filtered_portfolio = filtered_portfolio.resample('D', on='Date').mean().reset_index()
    filtered_portfolio['Date'] = pd.to_datetime(filtered_portfolio['Date'])

# Define theme-aware font color
font_color = '#000000' if st.session_state.theme == 'light' else '#ffffff'

# Price trend plot (enhanced line plot)
fig_price = px.line(filtered_predictions, x='Date', y=['Actual_Close', 'Predicted_Close'],
                    title="AAPL Price Trends with Buy/Sell Signals",
                    labels={'value': 'Price ($)', 'variable': 'Price Type'})
fig_price.update_traces(line=dict(width=3))
fig_price.update_traces(selector=dict(name='Actual_Close'), line=dict(color='#1f77b4'))
fig_price.update_traces(selector=dict(name='Predicted_Close'), line=dict(color='#ff7f0e'))
if not filtered_trades.empty:
    buy_signals = go.Scatter(
        x=filtered_trades['Buy_Date'],
        y=filtered_trades['Buy_Price'],
        mode='markers',
        name='Buy Signal',
        marker=dict(symbol='triangle-up', size=12, color='green')
    )
    fig_price.add_trace(buy_signals)
    if 'Sell_Date' in filtered_trades.columns and not filtered_trades['Sell_Date'].isna().all():
        sell_signals = go.Scatter(
            x=filtered_trades['Sell_Date'],
            y=filtered_trades['Sell_Price'],
            mode='markers',
            name='Sell Signal',
            marker=dict(symbol='triangle-down', size=12, color='red')
        )
        fig_price.add_trace(sell_signals)
    else:
        st.warning("Sell signals not plotted due to missing or invalid Sell_Date.")
fig_price.update_layout(
    hovermode='x unified',
    plot_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c',
    paper_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c',
    font=dict(size=12, color=font_color),
    xaxis=dict(title_font=dict(color=font_color), tickfont=dict(color=font_color)),
    yaxis=dict(title_font=dict(color=font_color), tickfont=dict(color=font_color)),
    showlegend=True
)
st.plotly_chart(fig_price, use_container_width=True)

# Day-wise Profit Margin Line Chart
if not filtered_trades.empty:
    profit_margin_df = filtered_trades.copy()
    profit_margin_df['Profit Margin (%)'] = ((profit_margin_df['Sell_Price'] - profit_margin_df['Buy_Price']) / profit_margin_df['Buy_Price']) * 100
    profit_margin_df['Date'] = profit_margin_df['Buy_Date'].dt.date
    profit_margin_daily = profit_margin_df.groupby('Date')['Profit Margin (%)'].sum().reset_index()
    profit_margin_daily['Date'] = pd.to_datetime(profit_margin_daily['Date'])
    if not profit_margin_daily.empty:
        fig_profit = px.line(
            profit_margin_daily,
            x='Date',
            y='Profit Margin (%)',
            title="Day-wise Profit Margin Trend",
            labels={'Profit Margin (%)': 'Profit Margin (%)'},
            color_discrete_sequence=['#1f77b4']
        )
        fig_profit.update_traces(
            line=dict(width=3),
            mode='lines+markers',
            marker=dict(size=8, symbol='circle'),
            hovertemplate='Date: %{x|%Y-%m-%d}<br>Profit Margin: %{y:.2f}%'
        )
        fig_profit.update_layout(
            hovermode='x unified',
            plot_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c',
            paper_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c',
            font=dict(size=12, color=font_color),
            xaxis=dict(title_font=dict(color=font_color), tickfont=dict(color=font_color)),
            yaxis=dict(title_font=dict(color=font_color), tickfont=dict(color=font_color)),
            showlegend=False
        )
        st.plotly_chart(fig_profit, use_container_width=True)
    else:
        st.warning("No profit margin data available for the selected date range.")
else:
    st.warning("No profit margin data available for the selected date range.")

# Portfolio value plot with markers
if not filtered_portfolio.empty:
    fig_portfolio = go.Figure()
    fig_portfolio.add_trace(
        go.Scatter(
            x=filtered_portfolio['Date'],
            y=filtered_portfolio['Portfolio_Value'],
            mode='lines+markers',
            name='Portfolio Value',
            line=dict(color='green', width=3),
            marker=dict(symbol='circle', size=8, color='green')
        )
    )
    fig_portfolio.update_layout(
        title="Portfolio Value Over Time",
        xaxis_title="Date",
        yaxis_title="Value ($)",
        hovermode='x unified',
        plot_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c',
        paper_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c',
        font=dict(size=12, color=font_color),
        xaxis=dict(title_font=dict(color=font_color), tickfont=dict(color=font_color)),
        yaxis=dict(title_font=dict(color=font_color), tickfont=dict(color=font_color)),
        showlegend=True
    )
    st.plotly_chart(fig_portfolio, use_container_width=True)
else:
    st.warning("No portfolio data available for the selected date range.")

# Non-graph sections (shown only in Full Dashboard mode)
if nav_option == "Full Dashboard":
    # Buy/Sell Records Table
    if not filtered_trades.empty:
        records_df = filtered_trades[['Buy_Date', 'Buy_Price', 'Sell_Date', 'Sell_Price', 'Profit']].copy()
        records_df['Profit Margin (%)'] = ((records_df['Sell_Price'] - records_df['Buy_Price']) / records_df['Buy_Price']) * 100
        records_df['Buy_Date'] = records_df['Buy_Date'].dt.strftime('%Y-%m-%d %H:%M')
        records_df['Sell_Date'] = records_df['Sell_Date'].dt.strftime('%Y-%m-%d %H:%M')
        records_df = records_df.rename(columns={
            'Buy_Date': 'Buy Date (Hour)',
            'Buy_Price': 'Buy Price ($)',
            'Sell_Date': 'Sell Date (Hour)',
            'Sell_Price': 'Sell Price ($)',
            'Profit': 'Profit ($)',
            'Profit Margin (%)': 'Profit Margin (%)'
        })
        st.subheader("Buy/Sell Records")
        st.dataframe(records_df, use_container_width=True)
    else:
        st.warning("No trade records available for the selected date range.")

    # Performance metrics
    if not filtered_trades.empty and not filtered_portfolio.empty:
        total_profit = filtered_trades['Profit'].sum()
        win_rate = (trades_df['Profit'] > 0).mean() * 100
        initial_cash = 10000
        final_value = filtered_portfolio['Portfolio_Value'].iloc[-1]
        growth = ((final_value - initial_cash) / initial_cash) * 100
        cash_ratio = filtered_portfolio['Cash'].iloc[-1] / final_value
        diversification = "Balanced" if 0.4 <= cash_ratio <= 0.6 else "Unbalanced"
        returns = filtered_trades['Profit'] / filtered_trades['Buy_Price']
        sharpe_ratio = np.mean(returns) / np.std(returns) * np.sqrt(252 * 6) if len(returns) > 1 else 0
        performance = "Strong" if growth > 2 else "Moderate" if growth > 0 else "Loss"
        
        metrics_df = pd.DataFrame({
            'Metric': ['Total Profit ($)', 'Win Rate (%)', 'Growth (%)', 'Diversification', 'Sharpe Ratio', 'Performance'],
            'Value': [f"{total_profit:.2f}", f"{win_rate:.1f}", f"{growth:.2f}", diversification, f"{sharpe_ratio:.2f}", performance]
        })
        st.subheader("Performance Metrics")
        st.dataframe(metrics_df, use_container_width=True)
    else:
        st.warning("No trades or portfolio data available for the selected date range.")

    # Visual Representation section
    st.subheader("Visual Representation")
    
    # Stock Held by Users
    with st.expander("Stock Held by Users"):
        if not filtered_trades.empty:
            shares_df = pd.DataFrame()
            buy_shares = filtered_trades[['Buy_Date']].copy()
            buy_shares['Shares'] = 1
            buy_shares['Type'] = 'Buy'
            sell_shares = filtered_trades[['Sell_Date']].copy()
            sell_shares['Shares'] = -1
            sell_shares['Type'] = 'Sell'
            sell_shares = sell_shares.rename(columns={'Sell_Date': 'Buy_Date'})
            shares_df = pd.concat([buy_shares, sell_shares], ignore_index=True)
            shares_df['Date'] = pd.to_datetime(shares_df['Buy_Date']).dt.date
            shares_df = shares_df.groupby('Date')['Shares'].sum().reset_index()
            shares_df['Date'] = pd.to_datetime(shares_df['Date'])
            shares_df['Cumulative Shares'] = shares_df['Shares'].cumsum()
            
            fig_shares = px.bar(
                shares_df,
                x='Date',
                y='Cumulative Shares',
                title="Stock Held by Users (Cumulative Shares)",
                labels={'Cumulative Shares': 'Shares Held'},
                color_discrete_sequence=['#1f77b4']
            )
            fig_shares.update_traces(
                hovertemplate='Date: %{x|%Y-%m-%d}<br>Shares Held: %{y}'
            )
            fig_shares.update_layout(
                hovermode='x unified',
                plot_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c',
                paper_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c',
                font=dict(size=12, color=font_color),
                xaxis=dict(title_font=dict(color=font_color), tickfont=dict(color=font_color)),
                yaxis=dict(title_font=dict(color=font_color), tickfont=dict(color=font_color)),
                showlegend=False
            )
            st.plotly_chart(fig_shares, use_container_width=True)
        else:
            st.warning("No trade data available for stock holdings.")
    
    # Profit Share (Hourly/Monthly)
    with st.expander("Profit Share (Hourly/Monthly)"):
        if not filtered_trades.empty:
            time_frame = st.selectbox("Select Time Frame", ["Hourly", "Monthly"], key="profit_share_time")
            profit_df = filtered_trades[['Buy_Date', 'Profit']].copy()
            profit_df['Date'] = pd.to_datetime(profit_df['Buy_Date'])
            if time_frame == "Hourly":
                profit_df = profit_df.resample('H', on='Date')['Profit'].sum().reset_index()
            else:
                profit_df = profit_df.resample('M', on='Date')['Profit'].sum().reset_index()
            
            fig_profit_share = px.area(
                profit_df,
                x='Date',
                y='Profit',
                title=f"Profit Share ({time_frame})",
                labels={'Profit': 'Profit ($)'},
                color_discrete_sequence=['#ff7f0e']
            )
            fig_profit_share.update_traces(
                hovertemplate='Date: %{x|%Y-%m-%d %H:%M}<br>Profit: $%{y:.2f}' if time_frame == "Hourly" else 'Date: %{x|%Y-%m}<br>Profit: $%{y:.2f}'
            )
            fig_profit_share.update_layout(
                hovermode='x unified',
                plot_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c',
                paper_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c',
                font=dict(size=12, color=font_color),
                xaxis=dict(title_font=dict(color=font_color), tickfont=dict(color=font_color)),
                yaxis=dict(title_font=dict(color=font_color), tickfont=dict(color=font_color)),
                showlegend=False
            )
            st.plotly_chart(fig_profit_share, use_container_width=True)
        else:
            st.warning("No trade data available for profit share.")
    
    # Stock Value Rises (Hourly/Daily)
    with st.expander("Stock Value Rises (Hourly/Daily)"):
        if not filtered_predictions.empty:
            time_frame = st.selectbox("Select Time Frame", ["Hourly", "Daily"], key="value_rises_time")
            value_df = filtered_predictions[['Date', 'Actual_Close']].copy()
            value_df['Price Change'] = value_df['Actual_Close'].diff()
            if time_frame == "Daily":
                value_df = value_df.resample('D', on='Date').mean().reset_index()
                value_df['Price Change'] = value_df['Actual_Close'].diff()
            
            fig_value_rises = px.line(
                value_df,
                x='Date',
                y='Price Change',
                title=f"Stock Value Rises ({time_frame})",
                labels={'Price Change': 'Price Change ($)'},
                color_discrete_sequence=['#1f77b4']
            )
            fig_value_rises.update_traces(
                line=dict(width=3),
                mode='lines+markers',
                marker=dict(size=8, symbol='circle'),
                hovertemplate='Date: %{x|%Y-%m-%d %H:%M}<br>Price Change: $%{y:.2f}' if time_frame == "Hourly" else 'Date: %{x|%Y-%m-%d}<br>Price Change: $%{y:.2f}'
            )
            fig_value_rises.update_layout(
                hovermode='x unified',
                plot_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c',
                paper_bgcolor='#ffffff' if st.session_state.theme == 'light' else '#2c2c2c',
                font=dict(size=12, color=font_color),
                xaxis=dict(title_font=dict(color=font_color), tickfont=dict(color=font_color)),
                yaxis=dict(title_font=dict(color=font_color), tickfont=dict(color=font_color)),
                showlegend=False
            )
            st.plotly_chart(fig_value_rises, use_container_width=True)
        else:
            st.warning("No price data available for stock value rises.")